# 02 · Off-chain evidence tutorial — Shilin

This eight-part notebook presents Shilin's prespecified extension to the 2026-09-14 12:00–12:05 UTC bounded pilot. It uses the **same 116 creation events** in [Claire's fixed on-chain release](https://huggingface.co/datasets/global-nomad-nexus/claire-threechain-v1/tree/8b29598a6565b67a8a943962dbf77f3d6b2559de); it adds source observations, not new events. The protocol was committed before extension collection (`92c2df1`). Independent coauthor review is pending.

[Protocol](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/EXTENSION_PROTOCOL.md) · [Data dictionary](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/DATA_DICTIONARY.md) · [V1 linkage tables](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v1/release) · [V2 extension tables](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/release). Run the code cells in order in a CPU runtime. Published tables omit third-party response bodies pending rights review.

<a id="part-1"></a>
## Part 1 — Research question and navigation

What can be documented when a chain-declared metadata URI is observed after token creation? We preserve event, request, snapshot, field, and image-resource units separately. A field URL is a declaration in retrieved JSON; its destination and ownership require further evidence. The later retrieval cannot establish creation-time availability.

1. [Question](#part-1) · 2. [Sources](#part-2) · 3. [Acquisition](#part-3) · 4. [Field audit](#part-4) · 5. [Processing](#part-5) · 6. [Coverage](#part-6) · 7. [Descriptive result](#part-7) · 8. [Validation](#part-8).

<a id="part-2"></a>
## Part 2 — Sources, units and diagram

Claire's [unchanged release](https://huggingface.co/datasets/global-nomad-nexus/claire-threechain-v1/tree/8b29598a6565b67a8a943962dbf77f3d6b2559de) covers 2026-09-14 12:00–12:05 UTC. Shilin v1 selected 116 eligible creation events: 61 Pump.fun, 55 Four.meme, and zero recognized Clanker creations. V2 uses all 57 distinct Pump JSON snapshots, including those without website fields. The unit may be an event, response, field, image URI or request; each table in the [dictionary](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/DATA_DICTIONARY.md) says which.

<img src="https://raw.githubusercontent.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/0655881/pilots/shilin-offchain-v1/figures/pipeline.svg" width="900" alt="Source-to-linkage pipeline" />

```mermaid
flowchart LR
 A[Claire creation event] --> B[Declared metadata URI]
 B --> C[Shilin v1 JSON snapshot]
 B --> D[Second metadata request]
 C --> E[Field audit]
 C --> F[Declared image URI]
 F --> G[Image request and digest]
 E --> H[Review flags and coverage]
 D --> H
 G --> H
```

Each arrow is a recorded source or processing step. Chain event time and off-chain retrieval time are different.

In [ ]:
import os, sys, json, tarfile, tempfile, urllib.request
from pathlib import Path
from collections import Counter
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow>=23,<26"])
    import pyarrow as pa
    import pyarrow.parquet as pq

COMMIT = "0655881"
local = os.environ.get("PILOT_LOCAL_REPO")
if local:
    ROOT = Path(local).expanduser().resolve()
    print("Using local author checkout:", ROOT)
else:
    url = f"https://codeload.github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/tar.gz/{COMMIT}"
    request = urllib.request.Request(url, headers={"User-Agent": "shilin-offchain-v2-colab/1.0"})
    body = urllib.request.urlopen(request, timeout=90).read()
    temp = Path(tempfile.mkdtemp(prefix="shilin-v2-"))
    archive = temp / "input.tar.gz"
    archive.write_bytes(body)
    with tarfile.open(archive, "r:gz") as tar:
        prefix = tar.getnames()[0].split("/")[0]
        tar.extractall(temp, filter="data")
    ROOT = temp / prefix
V1 = ROOT / "pilots/shilin-offchain-v1/release"
V2 = ROOT / "pilots/shilin-offchain-v2/release"
sys.path.insert(0, str(ROOT / "pilots/shilin-offchain-v2"))
from verify_extension import verify
result = verify(V2, V1)
assert result["passed"], result
print("Fixed public release verified; PyArrow", pa.__version__)

<a id="part-3"></a>
## Part 3 — Acquire and preserve source observations

The [prespecified protocol](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/EXTENSION_PROTOCOL.md) requested every one of the 57 metadata URIs a second time and every distinct image URI in their fixed JSON, once each. The [collector](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/collect_images.py) logged exact URL, UTC, status, redirects, media type, byte count and SHA-256; image responses were capped at 2 MiB. Third-party response bodies remain local until redistribution rights are settled. The cells below inspect **fixed real request receipts** and do not silently replace them with today's response.

In [ ]:
metadata = pq.read_table(V2 / "metadata_refetch.parquet").to_pylist()
images = pq.read_table(V2 / "image_acquisition.parquet").to_pylist()
print("Metadata requests:", len(metadata), dict(Counter(r["status"] for r in metadata)))
print("Image requests:", len(images), dict(Counter(r["status"] for r in images)))
print("Example metadata receipt:", {k:metadata[0][k] for k in ("original_uri","retrieved_at_utc","status_code","raw_sha256","same_bytes_as_v1")})
assert len(metadata)==57 and len(images)==56

<a id="part-4"></a>
## Part 4 — Inspect field observations

The 57 v1 JSON snapshots each have 14 prespecified field rows, even if the field is absent. `nonempty`, `empty`, `null`, `absent`, and `other_type` are distinct states. `showName` is often Boolean, so `other_type` is not automatically an error. Only specified URL values are in the public rights-limited table.

In [ ]:
fields = pq.read_table(V2 / "metadata_field_audit.parquet").to_pylist()
for field in ("image","coin_community","website","twitter","telegram","description"):
    rows=[r for r in fields if r["field_name"]==field]
    print(field, dict(Counter(r["field_state"] for r in rows)))
print("Second retrieval same bytes:", sum(r["same_bytes_as_v1"] is True for r in metadata), "/", len(metadata))
assert len(fields)==798

<a id="part-5"></a>
## Part 5 — Processing and linkage semantics

The [prespecified extraction](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/prepare_extension.py) checks the retained v1 response hashes before auditing 14 fields per snapshot. It compares literal `name` and `symbol` values against Claire's decoded creation fields for all 61 Pump events. The original 39 website/social URL declarations and typed assertions stay in v1; v2 adds separate image and `coin_community` observations. A social post in a `twitter` field remains a post URL. The [freeze step](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/freeze_extension.py) checks response bytes and image signatures; its CID check applies only to single-block CIDv1 raw SHA-256 objects.

In [ ]:
event_checks = pq.read_table(V2 / "event_metadata_checks.parquet").to_pylist()
url_checks = pq.read_table(V2 / "url_semantic_checks.parquet").to_pylist()
print("Name matches:", sum(r["name_exact_match"] is True for r in event_checks), "/", len(event_checks))
print("Symbol matches:", sum(r["symbol_exact_match"] is True for r in event_checks), "/", len(event_checks))
print("URL semantic states:", dict(Counter(r["semantic_state"] for r in url_checks)))
print("Image signatures:", dict(Counter(str(r["media_magic"]) for r in images)))
print("Image CID checks:", dict(Counter(str(r["cid_integrity_status"]) for r in images)))
assert len(event_checks)==61 and len(url_checks)==39
v1_declarations = pq.read_table(V1 / "offchain_declarations.parquet").to_pylist()
print("Original URL declarations retained:", len(v1_declarations))
assert len(v1_declarations)==39


<a id="part-6"></a>
## Part 6 — Inspect linked data and uncertainty

All 116 creation events remain in the coverage table. The 55 Four.meme events have no new off-chain source in this extension; their v1 access states remain visible. Image acquisition failures remain failures.

In [ ]:
coverage = pq.read_table(V2 / "event_extension_coverage.parquet").to_pylist()
print("Events by platform:", dict(Counter(r["platform_id"] for r in coverage)))
print("V1 linkage coverage:", dict(Counter(r["v1_coverage_state"] for r in coverage)))
print("V2 image states:", dict(Counter(r["image_acquisition_state"] for r in coverage)))
assert len(coverage)==116

<a id="part-7"></a>
## Part 7 — Small descriptive example

Count source outcomes while keeping every event in the denominator. These are observations from one five-minute cohort, not estimates of all token launches or effects of social presence. [EX-Graph](https://proceedings.iclr.cc/paper_files/paper/2024/hash/ba29e3f830d039c3f1fa0b4dfcf19c54-Abstract-Conference.html) already links Ethereum and X; our proposed difference is the evidence and time record. [Multi-Chain Graphs of Graphs](https://proceedings.neurips.cc/paper_files/paper/2024/hash/3205b048f9cc54b9f7963db0b0f52d53-Abstract-Datasets_and_Benchmarks_Track.html) motivates explicit chain-specific denominators.

In [ ]:
for platform in sorted({r["platform_id"] for r in coverage}):
    rows=[r for r in coverage if r["platform_id"]==platform]
    print(platform, {"events":len(rows), "metadata":dict(Counter(r["metadata_refetch_state"] for r in rows)), "image":dict(Counter(r["image_acquisition_state"] for r in rows))})
assert sum(r["status"]=="response_saved" for r in images)==54

<a id="part-8"></a>
## Part 8 — Technical validation and handoff

The measured validation report in this pilot directory distinguishes 57 unchanged metadata response digests, 54 obtainable image bodies, and two responses over the 2 MiB cap. Thirty single-block CIDv1 raw-image digests matched; other CID forms were not verified by this method. The image source rights register lists image-serving hosts, not image owners. Raw JSON and image bytes remain outside the public release. Formal publication requires a new frozen publication cohort, rights decisions, and independent adjudication; no precision, recall, ownership, or launch-time site claim follows from this pilot.

In [ ]:
final = verify(V2, V1)
validation = json.loads((V2 / "validation.json").read_text())
print("Checks passed:", sum(final["checks"].values()), "/", len(final["checks"]))
print("Metadata retrievals:", validation["metadata_refetch_statuses"])
print("Image retrievals:", validation["image_acquisition_statuses"])
assert final["passed"]